# Export a toy model to ONNX

The detector is trained in TensorFlow but runs on ONNX Runtime (see ADR 0002).
This notebook shows every step on a tiny model, so you can see what `make export-detector` does:

1. Build a small Keras model.
2. Save it as a TensorFlow SavedModel.
3. Convert the SavedModel to an `.onnx` file with tf2onnx.
4. Look at the ONNX inputs and outputs.
5. Run the `.onnx` file with ONNX Runtime and check it gives the same answers as Keras.

Run it with `make notebook` (it needs the `train` extra and the `notebook` group).

## 1. Build a tiny model

The model takes an 8×8 RGB picture and gives two scores that add up to 1.
It is not trained; random weights are enough to check the export.
The seed makes the numbers the same every run.

In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's start-up messages

import tempfile
from pathlib import Path

import keras
import numpy as np
import tensorflow as tf

tf.get_logger().setLevel("ERROR")
keras.utils.set_random_seed(0)
model = keras.Sequential(
    [
        keras.Input(shape=(8, 8, 3), name="image"),
        keras.layers.Conv2D(4, 3, activation="relu"),
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(2, activation="softmax"),
    ]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 6, 6, 4)        │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 4)              │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │            10 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122 (488.00 B)

 Trainable params: 122 (488.00 B)

 Non-trainable params: 0 (0.00 B)

## 2. Save it as a SavedModel

tf2onnx reads a TensorFlow SavedModel folder. `model.export` writes one.
The TF model zoo detectors that `make export-detector` downloads are SavedModels too.
Files go to a temporary folder so nothing is written into the repo.

In [2]:
work = Path(tempfile.mkdtemp())
saved_model = work / "saved_model"
model.export(saved_model, verbose=False)
sorted(path.name for path in saved_model.iterdir())

['assets', 'fingerprint.pb', 'saved_model.pb', 'variables']

## 3. Convert it to ONNX

This uses the same command as `make export-detector`: `catpoop.export_detector.tf2onnx_command`
builds `python -m tf2onnx.convert --saved-model ... --opset 17 --output ...`.
The opset is the version of the ONNX operator set; ONNX Runtime must support it.

In [3]:
import subprocess

from catpoop.export_detector import OPSET, tf2onnx_command

onnx_file = work / "toy.onnx"
command = tf2onnx_command(saved_model, onnx_file)
print(" ".join(command[1:]).replace(str(work), "<work>"))
subprocess.run(command, check=True, capture_output=True)
print(f"opset {OPSET}, {onnx_file.stat().st_size} bytes")

-m tf2onnx.convert --saved-model <work>/saved_model --opset 17 --output <work>/toy.onnx


opset 17, 2426 bytes


## 4. Look inside the ONNX file

`onnx.checker` checks the file is valid. The inputs and outputs are what ONNX Runtime will ask for and give back.
A shape like `unk__12` is a size that can change (here, the batch size).
tf2onnx keeps the input name (`image`) and names the output `output_0`.

In [4]:
import onnx

onnx_model = onnx.load(onnx_file)
onnx.checker.check_model(onnx_model)
print("opsets:", {item.domain or "ai.onnx": item.version for item in onnx_model.opset_import})


def describe(value):
    """The name and shape of an ONNX input or output."""
    dims = [dim.dim_value or dim.dim_param for dim in value.type.tensor_type.shape.dim]
    return f"{value.name}: {dims}"


print("inputs: ", [describe(value) for value in onnx_model.graph.input])
print("outputs:", [describe(value) for value in onnx_model.graph.output])

opsets: {'ai.onnx': 17, 'ai.onnx.ml': 2}
inputs:  ["image: ['unk__12', 8, 8, 3]"]
outputs: ["output_0: ['unk__13', 2]"]


## 5. Run it with ONNX Runtime and compare with Keras

`catpoop.onnx_session.open_session` is how the app opens the detector; `runtime="cpu"` runs it on the CPU.
Both models get the same two random pictures. The answers should match to about 6 decimal places
(small differences come from floating point rounding).

In [5]:
from catpoop.onnx_session import open_session

session = open_session(onnx_file, runtime="cpu")
for port in session.get_inputs():
    print("input ", port.name, port.shape, port.type)
for port in session.get_outputs():
    print("output", port.name, port.shape, port.type)

pictures = np.random.default_rng(0).random((2, 8, 8, 3), dtype=np.float32)
from_keras = model.predict(pictures, verbose=0)
(from_onnx,) = session.run(None, {session.get_inputs()[0].name: pictures})
print("keras:", from_keras.round(6).tolist())
print("onnx: ", from_onnx.round(6).tolist())
np.testing.assert_allclose(from_onnx, from_keras, rtol=1e-5, atol=1e-6)
print("max difference:", float(np.abs(from_onnx - from_keras).max()))

input  image ['unk__12', 8, 8, 3] tensor(float)
output output_0 ['unk__13', 2] tensor(float)


keras: [[0.468425989151001, 0.531574010848999], [0.46698901057243347, 0.5330110192298889]]
onnx:  [[0.468425989151001, 0.531574010848999], [0.46698901057243347, 0.5330110192298889]]
max difference: 5.960464477539063e-08


## The same steps for the real detector

| This notebook | The pet detector |
|---|---|
| `model.export(saved_model)` | The TF model zoo ships a SavedModel (or pass your own with `SAVED_MODEL=`) |
| `tf2onnx_command(...)` | `make export-detector OUT=/mnt/nas/cat-poop/models/pets.onnx` |
| `open_session(onnx_file, runtime="cpu")` | `[detector] runtime` in `config.toml` (`auto`, `cpu`, `amd` or `nvidia`) |
| The `.onnx` file in a temporary folder | `[detector] model`, relative to `paths.private_data_store` |
| Inputs and outputs from step 4 | The "onnx" detector wants one `[1, H, W, 3]` picture and the outputs `detection_boxes`, `detection_classes`, `detection_scores` and `num_detections` |

A model you train yourself must keep those output names, and its class ids go in `[detector.labels]`.
Check them with step 4 before you point `[detector] model` at the file.